# 09 · How a transformer reads text

The next three methods (IA3, prompt tuning, prefix tuning) all add knobs *inside*
the transformer — at the **token sequence** or at **attention**. We haven't met
those yet, so this notebook builds just enough, with tiny numbers. No method here —
pure primitive.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

## Step 1 — Text becomes a *sequence of vectors*

A model can't read words. Each token (a word or word-piece) is looked up in an
**embedding table** and replaced by a vector of numbers — its **embedding**. A
sentence is then a **sequence** of these vectors: a matrix of shape
`(n_tokens, d_features)`.

In [3]:
embed = rng.normal(size=(5, 4))    # tiny embedding table: 5 possible tokens, 4 features
sentence_ids = [2, 0, 3]           # a 3-token sentence (ids into the table)
seq = embed[sentence_ids]          # look each one up

print("sequence shape:", seq.shape, " -> 3 tokens, each a 4-feature vector")
print(embed)
print(seq.round(2))

sequence shape: (3, 4)  -> 3 tokens, each a 4-feature vector
[[-0.12853466  1.36646347 -0.66519467  0.35151007]
 [ 0.90347018  0.0940123  -0.74349925 -0.92172538]
 [-0.45772583  0.22019512 -1.00961818 -0.20917557]
 [-0.15922501  0.54084558  0.21465912  0.35537271]
 [-0.65382861 -0.12961363  0.78397547  1.49343115]]
[[-0.46  0.22 -1.01 -0.21]
 [-0.13  1.37 -0.67  0.35]
 [-0.16  0.54  0.21  0.36]]


## Step 2 — Attention: tokens look at each other

Inside each layer, tokens share information via **attention**. Every token builds
three vectors:

- a **Query** (Q): "what am I looking for?"
- a **Key** (K): "what do I represent?"
- a **Value** (V): "what will I pass on if attended to?"

A token compares its Query to every Key (a similarity score), turns those scores
into mixing weights (softmax), and outputs a **weighted blend of the Values**. Here
it is end to end on our 3-token sequence:

In [4]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

d = 4
Wq, Wk, Wv = rng.normal(size=(3, d, d))     # projections to Q, K, V
Q, K, V = seq @ Wq, seq @ Wk, seq @ Wv

scores  = Q @ K.T / np.sqrt(d)              # token-to-token similarity (3 x 3)
weights = softmax(scores)                   # each row sums to 1: a token's focus
out     = weights @ V                       # each output = a blend of the Values

print("attention weights (row i = token i's focus over all 3 tokens):")
print(weights.round(2))
print("output shape:", out.shape, " -> still 3 tokens x 4 features")

attention weights (row i = token i's focus over all 3 tokens):
[[0.05 0.52 0.43]
 [0.01 0.1  0.89]
 [0.2  0.23 0.57]]
output shape: (3, 4)  -> still 3 tokens x 4 features


That's the whole engine: **Keys and Values are the handles other tokens grab onto.**
Hold that — two of the next methods work by adding or rescaling K and V.

## Step 3 — Two places to inject new knobs

A frozen transformer gives us two natural spots to bolt on trainable pieces:

1. **At the input** — prepend extra trainable vectors to the *sequence* (Step 1).
   → that's **prompt tuning** (notebook 11).
2. **Inside attention, at every layer** — add or rescale **K / V** (Step 2).
   → add trainable K/V = **prefix tuning** (notebook 12); rescale K/V/FFN = **IA3**
   (notebook 10).

## Recap

- Text → tokens → **embeddings** → a `(n_tokens, d)` **sequence** of vectors.
- **Attention**: each token uses a **Query** to match others' **Keys**, then blends
  their **Values**.
- New knobs can go **at the input** (prompt) or **inside attention's K/V** (prefix,
  IA3).

**BAM!** Next: **`10 — IA3`.**